In [1]:
!pip install -q transformers sentence-transformers faiss-cpu gradio pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.5 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import requests
import os

url = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"

save_path = "/content/drive/MyDrive/threat_chatbot/enterprise-attack.json"  # 🔴 CHANGE PATH HERE

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(save_path), exist_ok=True)

response = requests.get(url)

with open(save_path, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")

Downloaded successfully!


In [5]:
import json
import pandas as pd

json_path = "/content/drive/MyDrive/threat_chatbot/enterprise-attack.json"

with open(json_path) as f:
    data = json.load(f)

rows = []

for obj in data['objects']:
    if obj.get('type') == 'attack-pattern':
        name = obj.get('name', '')
        description = obj.get('description', '')
        rows.append([name, description])

df = pd.DataFrame(rows, columns=['technique', 'description'])

df.to_csv("/content/drive/MyDrive/threat_chatbot/mitre_data.csv", index=False)

print("CSV ready!")

CSV ready!


In [6]:
FAISS_PATH = "/content/drive/MyDrive/threat_chatbot/faiss_index"
DATA_PATH = "/content/drive/MyDrive/threat_chatbot/mitre_data.csv"

In [7]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

# Assume columns: technique, description
documents = df['description'].tolist()

In [8]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

all_chunks = []

for doc in documents:
    all_chunks.extend(chunk_text(doc))

print("Total Chunks:", len(all_chunks))

Total Chunks: 940


In [9]:
from sentence_transformers import SentenceTransformer

# MiniLM model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embed_model.encode(all_chunks, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

In [10]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

FAISS index created with 940 vectors


In [11]:
import pickle
import os

os.makedirs(FAISS_PATH, exist_ok=True)

# Save index
faiss.write_index(index, FAISS_PATH + "/index.faiss")

# Save chunks
with open(FAISS_PATH + "/chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Saved to Google Drive")

Saved to Google Drive


In [12]:
# Load index
index = faiss.read_index(FAISS_PATH + "/index.faiss")

# Load chunks
with open(FAISS_PATH + "/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

In [13]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",   #  CHANGE MODEL HERE if needed
    max_length=512
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

In [14]:
def retrieve(query, k=3):  # CHANGE k HERE

    query_embedding = embed_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = [all_chunks[i] for i in indices[0]]

    return results

In [15]:
def rag_pipeline(query):

    retrieved_docs = retrieve(query, k=3)

    context = " ".join(retrieved_docs)

    prompt = f"""
    You are a cybersecurity threat intelligence assistant.

    Context:
    {context}

    Question:
    {query}

    Answer clearly:
    """

    response = llm(prompt)[0]['generated_text']

    return response

In [17]:
import gradio as gr

def chatbot(query):
    return rag_pipeline(query)

with gr.Blocks(theme=gr.themes.Soft()) as interface:

    gr.Markdown("""
    # Threat Intelligence Chatbot
    Ask cybersecurity questions based on MITRE ATT&CK dataset
    """)

    query_input = gr.Textbox(
        label="Enter Your Question",
        placeholder="Example: Explain T1059 technique",
        lines=3
    )

    output_box = gr.Textbox(
        label="Chatbot Response",
        lines=20,          # 🔥 Increase output size
        max_lines=30,
        show_copy_button=True
    )

    submit_btn = gr.Button("Submit")

    submit_btn.click(
        fn=chatbot,
        inputs=query_input,
        outputs=output_box
    )

interface.launch(share=True)

/tmp/ipykernel_4640/406127281.py:6: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a35baed7c8ec27617a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
